<a href="https://colab.research.google.com/github/busybee-123/Pollinator_Cam/blob/main/Custom_detection_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Script to run on HPC to run YOLO detection on the RTSP stream

In [ ]:
import csv
import os
import time
import logging
from datetime import datetime, time as dtime
import cv2
from ultralytics import YOLO

# Force FFmpeg to use TCP for RTSP stability
os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "rtsp_transport;tcp"

# --- ROI CONFIGURATION (From rpicam-vid / libcamera flags) ---
# Command flags used: --width 1920 --height 1080 --roi 0.1875,0.0556,0.6693,0.8380
STREAM_WIDTH = 1920
STREAM_HEIGHT = 1080

ROI_X = 0.2879
ROI_Y = 0.2557
ROI_W = 0.4685
ROI_H = 0.5866

# Calculate offsets relative to the 1920x1080 stream space
ROI_OFFSET_X = int((ROI_X / ROI_W) * STREAM_WIDTH)   # 537 px
ROI_OFFSET_Y = int((ROI_Y / ROI_H) * STREAM_HEIGHT)  # 71 px

# Interval interval in seconds (30 minutes)
INTERVAL_SECONDS = 30 * 60

# --- HELPER FUNCTIONS ---
def is_within_allowed_window():
    now = datetime.now().time()
    start_time = dtime(5, 30)   # 5:30 AM
    end_time = dtime(20, 30)    # 8:30 PM (20:30)
    return start_time <= now <= end_time

def is_duplicate_box(new_box, recent_boxes, pixel_tolerance=2):
    nx1, ny1, nx2, ny2 = new_box
    for (rx1, ry1, rx2, ry2) in recent_boxes:
        if (abs(nx1 - rx1) <= pixel_tolerance and
            abs(ny1 - ry1) <= pixel_tolerance and
            abs(nx2 - rx2) <= pixel_tolerance and
            abs(ny2 - ry2) <= pixel_tolerance):
            return True
    return False

# --- CONFIGURATION & DIRECTORIES ---
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f"/hpc/groups/pollinator-monitoring/runs/detect/run_{run_timestamp}"

crops_dir = os.path.join(output_dir, f"crops_{run_timestamp}")
os.makedirs(crops_dir, exist_ok=True)

# Folder for full frame snapshots triggered by detections
detection_full_images_dir = os.path.join(output_dir, f"detection_full_images_{run_timestamp}")
os.makedirs(detection_full_images_dir, exist_ok=True)

# Folder for full frame snapshots captured on 30-minute intervals
interval_full_images_dir = os.path.join(output_dir, f"interval_full_images_{run_timestamp}")
os.makedirs(interval_full_images_dir, exist_ok=True)

csv_path = os.path.join(output_dir, f"tracking_results_{run_timestamp}.csv")
log_path = os.path.join(output_dir, "stream_errors.log")

# Setup Logging
logging.basicConfig(
    filename=log_path,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logging.info("Tracking session initialized.")

# Initialize CSV file header if it doesn't exist
if not os.path.exists(csv_path):
    with open(csv_path, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Frame", "Track_ID", "Class_Name", "Confidence", "BBox_Local", "BBox_Global", "Crop_Path", "Full_Image_Path"])

# 1. Load YOLO Model
model = YOLO("weights/model_1.pt")
source_stream = "rtsp://157.140.134.43:8554/stream1"

saved_boxes_history = {}
last_interval_save_time = 0  # Initialize timer for 30-minute intervals

print(f"Tracking started. Lightweight crop extractor running. Output directory: {output_dir}")

# --- RECONNECT & TRACKING LOOP ---
while True:
    try:
        logging.info(f"Connecting to RTSP stream: {source_stream}")

        # Start tracking stream with 640p model sizing
        results = model.track(
            source=source_stream,
            stream=True,
            persist=True,
            show=False,
            conf=0.5,
            imgsz=640,          # Performs lightweight 640p inference while preserving original stream frame resolution
            save=False,         # Disables saving full video files
            save_crop=False
        )

        for frame_idx, r in enumerate(results):

            # Pause processing outside allowed window (5:30 AM - 8:30 PM)
            while not is_within_allowed_window():
                msg = "Outside allowed window (5:30 AM - 8:30 PM). Sleeping for 60s..."
                print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")
                logging.info(msg)
                time.sleep(60)

            if r.orig_img is not None:
                orig_frame = r.orig_img  # Stream cropped frame (1920x1080)
                h, w, _ = orig_frame.shape
                current_time = time.time()
                frame_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3]

                # --- 30-MINUTE INTERVAL SNAPSHOT LOGIC ---
                if current_time - last_interval_save_time >= INTERVAL_SECONDS:
                    interval_filename = f"interval_frame_{frame_idx}_{frame_timestamp}.jpg"
                    interval_image_path = os.path.join(interval_full_images_dir, interval_filename)
                    if cv2.imwrite(interval_image_path, orig_frame, [int(cv2.IMWRITE_JPEG_QUALITY), 95]):
                        logging.info(f"Saved 30-minute interval full frame: {interval_image_path}")
                        last_interval_save_time = current_time

                # Process detections
                if r.boxes is not None and len(r.boxes) > 0:
                    boxes = r.boxes.xyxy.cpu().numpy()
                    clss = r.boxes.cls.cpu().numpy().astype(int)
                    confs = r.boxes.conf.cpu().numpy()

                    track_ids = r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else [-1] * len(boxes)

                    # Track whether full detection frame has been saved for this specific frame event
                    detection_full_image_saved = False
                    detection_full_image_path = ""

                    # Open CSV once per frame to record valid detections
                    with open(csv_path, mode="a", newline="") as f:
                        writer = csv.writer(f)

                        for crop_idx, (box, track_id, cls, conf) in enumerate(zip(boxes, track_ids, clss, confs)):
                            class_name = model.names[cls]

                            # Local coordinates within the 1920x1080 ROI stream
                            lx1, ly1, lx2, ly2 = [int(coord) for coord in box]
                            lx1, ly1 = max(0, lx1), max(0, ly1)
                            lx2, ly2 = min(w, lx2), min(h, ly2)

                            # Global coordinates mapped back to full image canvas
                            gx1 = lx1 + ROI_OFFSET_X
                            gy1 = ly1 + ROI_OFFSET_Y
                            gx2 = lx2 + ROI_OFFSET_X
                            gy2 = ly2 + ROI_OFFSET_Y

                            history_key = f"track_{track_id}" if track_id != -1 else f"cls_{cls}"
                            if history_key not in saved_boxes_history:
                                saved_boxes_history[history_key] = []

                            current_box_local = (lx1, ly1, lx2, ly2)

                            if is_duplicate_box(current_box_local, saved_boxes_history[history_key], pixel_tolerance=2):
                                continue

                            if (lx2 - lx1) > 0 and (ly2 - ly1) > 0:
                                # Save full detection frame image once per unique detection event
                                if not detection_full_image_saved:
                                    full_filename = f"frame_{frame_idx}_{frame_timestamp}.jpg"
                                    detection_full_image_path = os.path.join(detection_full_images_dir, full_filename)
                                    if cv2.imwrite(detection_full_image_path, orig_frame, [int(cv2.IMWRITE_JPEG_QUALITY), 95]):
                                        detection_full_image_saved = True
                                    else:
                                        detection_full_image_path = ""  # Reset path if write failed

                                # Extract crop from the stream frame
                                crop_img = orig_frame[ly1:ly2, lx1:lx2]
                                track_str = f"id{track_id}" if track_id != -1 else "unconfirmed"
                                crop_filename = f"{class_name}_frame{frame_idx}_{track_str}_{frame_timestamp}_{crop_idx}.jpg"
                                crop_path = os.path.join(crops_dir, crop_filename)

                                success = cv2.imwrite(crop_path, crop_img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                                if success:
                                    saved_boxes_history[history_key].append(current_box_local)
                                    writer.writerow([
                                        frame_idx,
                                        track_id,
                                        class_name,
                                        f"{conf:.2f}",
                                        [lx1, ly1, lx2, ly2],
                                        [gx1, gy1, gx2, gy2],
                                        crop_path,
                                        detection_full_image_path
                                    ])

    except Exception as e:
        error_msg = f"Stream disconnected or exception occurred: {str(e)}"
        print(f"[ERROR] {error_msg}")
        logging.error(error_msg, exc_info=True)

        print("Retrying connection in 60 seconds...")
        time.sleep(60)

